# Demo 6: Delta Lake – ACID tranzakciók és Time Travel
## Open Table Format adatmérnöki szemmel

**Cél:** Megértjük a Delta Lake alapvető képességeit: ACID írások, transaction log,
time travel, schema evolution és MERGE/upsert – mindezt Spark nélkül,
a `deltalake` (delta-rs) csomag segítségével.

**Kapcsolódó diák:** Lakehouse architektúra, Open Table Formats (Delta, Iceberg, Hudi)

## 0. Előkészítés

### Miért Delta Lake és nem plain Parquet?

A Parquet kiváló columnar tárolóformátum, de önmagában **nem garantál ACID-ot**:
- Ha egy írás félbeszakad, a fájl korrupt maradhat
- Nincs verziókövető – nem lehet visszamenni egy korábbi állapothoz
- Konkurens írók felülírhatják egymás adatait

A **Delta Lake** egy nyílt tárolóréteg, amely Parquet adatfájlokra épít,
és egy **JSON transaction log**-ot (`_delta_log/`) ad hozzá:
- **Atomicitás**: a commit vagy teljesen sikerül, vagy semmilyen módosítás nem látható
- **Time travel**: bármely korábbi verzió visszakereshető
- **Schema evolution**: új oszlop hozzáadható meglévő adatok módosítása nélkül
- **MERGE**: SQL MERGE szemantikájú upsert

In [ ]:
import subprocess
subprocess.run(["pip", "install", "-q", "deltalake>=0.14,<0.18", "duckdb", "pandas", "pyarrow", "faker"], check=True)

import json, os, shutil
from pathlib import Path
from datetime import datetime, timedelta
import random
import pandas as pd
import pyarrow as pa
from deltalake import DeltaTable, write_deltalake
from faker import Faker
import duckdb

fake = Faker('hu_HU')
random.seed(42)
Faker.seed(42)

# Delta tábla helye
DELTA_PATH = "/tmp/delta_demo/orders"

# Tiszta állapot az újrafuttathatóság érdekében
if Path(DELTA_PATH).exists():
    shutil.rmtree(DELTA_PATH)
Path(DELTA_PATH).mkdir(parents=True, exist_ok=True)

import deltalake
print(f"deltalake: {deltalake.__version__}  |  DuckDB: {duckdb.__version__}  |  PyArrow: {pa.__version__}")
print(f"Delta tábla helye: {DELTA_PATH}")

## 1. rész: Delta tábla létrehozása – az első commit

### Mi az a Delta tábla?

Egy Delta tábla két részből áll:
1. **Parquet adatfájlok** – a tényleges adat, columnar formátumban
2. **`_delta_log/` könyvtár** – JSON commit fájlok sorozata (transaction log)

Minden írási művelet egy **új commit JSON fájlt** hoz létre a `_delta_log/`-ban.
A Delta Lake engine ezeket a fájlokat olvassa el, hogy meghatározza,
melyik Parquet fájlok tartoznak az aktuális (vagy egy adott verziójú) táblához.

In [ ]:
# Első adag: 2000 rendelés 2022–2023-ból
STATUSES = ["pending", "shipped", "delivered", "cancelled"]
CATEGORIES = ["Elektronika", "Laptop", "Telefon", "Butor", "Irodaszer", "Konyv"]

base = datetime(2022, 1, 1)
rows = []
for i in range(1, 2001):
    rows.append({
        "order_id":     i,
        "customer_id":  random.randint(1, 500),
        "product":      random.choice(CATEGORIES),
        "order_date":   (base + timedelta(days=random.randint(0, 730))).strftime("%Y-%m-%d"),
        "quantity":     random.randint(1, 10),
        "unit_price":   round(random.uniform(10, 500), 2),
        "order_status": random.choice(STATUSES),
    })

df_v0 = pd.DataFrame(rows)
df_v0["revenue"] = (df_v0["quantity"] * df_v0["unit_price"]).round(2)

# deltalake 0.14–0.17 pandas DataFrame-et közvetlenül elfogad
write_deltalake(DELTA_PATH, df_v0, mode="overwrite")

print(f"Version 0 letrehozva: {len(df_v0):,} sor")
print("\nDelta tabla konyvtarstruktura:")
for p in sorted(Path(DELTA_PATH).rglob('*'))[:15]:
    rel = p.relative_to(DELTA_PATH)
    indent = "  " * (len(rel.parts) - 1)
    print(f"  {indent}{rel.parts[-1]}")

### A `_delta_log` – a Delta Lake szíve

A `_delta_log/00000000000000000000.json` az első commit fájl. Tartalmaz:
- **protocol**: minimum reader/writer verzió
- **metaData**: séma (JSON Schema formátumban), táblanév, konfigurációk
- **add**: minden hozzáadott Parquet fájl neve, mérete, statisztikái (min/max/null count)

Ez a JSON fájl **soha nem módosul** – minden újabb commit egy új fájlt kap
(`00000000000000000001.json`, `00000000000000000002.json`, stb.).

In [ ]:
# A transaction log elso commit fajljanak megmutatasa
log_file = Path(DELTA_PATH) / "_delta_log" / "00000000000000000000.json"

print("=== _delta_log/00000000000000000000.json (reszlet) ===\n")
with open(log_file) as f:
    for line in f:
        entry = json.loads(line)
        action = list(entry.keys())[0]
        if action == "metaData":
            schema = json.loads(entry["metaData"]["schemaString"])
            print(f"[metaData] tabla ID: {entry['metaData']['id'][:8]}...")
            print(f"[metaData] oszlopok: {[f['name'] for f in schema['fields']]}")
        elif action == "add":
            print(f"[add]      fajl: {entry['add']['path'][:40]}...")
            print(f"[add]      meret: {entry['add']['size']:,} byte")
        elif action == "protocol":
            print(f"[protocol] minReaderVersion={entry['protocol']['minReaderVersion']}, "
                  f"minWriterVersion={entry['protocol']['minWriterVersion']}")

log_files = list((Path(DELTA_PATH) / "_delta_log").glob("*.json"))
print(f"\nJelenleg {len(log_files)} commit fajl a _delta_log/-ban")

## 2. rész: Append – második commit

### Append vs Overwrite

| Művelet | Mikor? | Régi adat? |
|---------|--------|------------|
| `mode="overwrite"` | Teljes tábla csere | Nem látható (de time travel-lel igen) |
| `mode="append"` | Új sorok hozzáadása | Megmarad |
| `mode="error"` | Alapértelmezett – hibát dob ha létezik | – |

Az `append` egy **új Parquet fájlt** ír és egy **új commit JSON-t** ad a `_delta_log`-ba.
A régi Parquet fájlok **nem módosulnak** – ez az immutabilitás elve.

In [ ]:
# Masodik adag: 500 uj rendelés 2024-bol
base2 = datetime(2024, 1, 1)
rows2 = []
for i in range(2001, 2501):
    rows2.append({
        "order_id":     i,
        "customer_id":  random.randint(1, 500),
        "product":      random.choice(CATEGORIES),
        "order_date":   (base2 + timedelta(days=random.randint(0, 365))).strftime("%Y-%m-%d"),
        "quantity":     random.randint(1, 10),
        "unit_price":   round(random.uniform(10, 500), 2),
        "order_status": random.choice(STATUSES),
    })

df_v1 = pd.DataFrame(rows2)
df_v1["revenue"] = (df_v1["quantity"] * df_v1["unit_price"]).round(2)

write_deltalake(DELTA_PATH, df_v1, mode="append")

print(f"Version 1 (append): +{len(df_v1):,} sor")
print(f"_delta_log fajlok: {len(list((Path(DELTA_PATH) / '_delta_log').glob('*.json')))}")

In [ ]:
# Transaction history – minden commit metaadatai
dt = DeltaTable(DELTA_PATH)
history = dt.history()

print("=== Delta tabla history ===")
for h in history:
    print(f"  version={h.get('version')}  "
          f"operation={h.get('operation'):<12}  "
          f"timestamp={str(h.get('timestamp'))[:19]}")

## 3. rész: Time Travel

### Miért értékes a time travel?

- **Rollback**: hibás betöltés után visszaállítás az előző verzióra
- **Audit**: ki, mikor, mit töltött be?
- **Reprodukálhatóság**: ML modell tanítása a 3 hónappal ezelőtti adatokon
- **Debugging**: összehasonlítás az előző és aktuális állapot között

A Delta Lake **nem törli a régi Parquet fájlokat** amíg `VACUUM` parancsot nem adunk ki.
Ezért bármely korábbi verziót vissza lehet tölteni.

In [ ]:
# Version 0: az eredeti 2000 sor (append elotti allapot)
df_read_v0 = DeltaTable(DELTA_PATH, version=0).to_pandas()

print(f"Version 0 sorok: {len(df_read_v0):,}")
print(f"Datum tartomany: {df_read_v0['order_date'].min()} -> {df_read_v0['order_date'].max()}")
print("\nMinta (elso 3 sor):")
print(df_read_v0[["order_id", "order_date", "product", "order_status", "revenue"]].head(3).to_string(index=False))

In [ ]:
# Version 1: az append utani allapot (2000 + 500 sor)
df_read_v1 = DeltaTable(DELTA_PATH, version=1).to_pandas()

print(f"Version 1 sorok: {len(df_read_v1):,}")
print(f"Datum tartomany: {df_read_v1['order_date'].min()} -> {df_read_v1['order_date'].max()}")
print("\nMinta (utolso 3 sor – 2024-es adat):")
print(df_read_v1[["order_id", "order_date", "product", "order_status", "revenue"]].tail(3).to_string(index=False))

In [ ]:
print("=== Time Travel osszehasonlitas ===")
print(f"  Version 0:  {len(df_read_v0):>6,} sor  (overwrite – 2022–2023)")
print(f"  Version 1:  {len(df_read_v1):>6,} sor  (append  – + 2024-es sorok)")
print(f"  Kulonbseg:  {len(df_read_v1) - len(df_read_v0):>6,} sor")
print()
print("-> A v0 adatok tovabbra is elerhetok – a Parquet fajlok nem toroldtek.")

## 4. rész: Schema Evolution

### Új oszlop hozzáadása meglévő táblához

Produkciós környezetben előfordul, hogy egy **új üzleti igény miatt új oszlop** kell.
Delta Lake kétféleképpen kezeli:

| Módszer | Leírás | Engine |
|---------|--------|--------|
| `schema_mode="merge"` | Addítív: csak az új oszlop kerül be, régi soroknál NULL | `engine="rust"` |
| `overwrite_schema=True` | Teljes sémacsere (override) | pyarrow vagy rust |

**Viselkedés merge esetén:**
- Régi Parquet fájlokban az új oszlop nem szerepel → NULL-ként jelenik meg
- Új Parquet fájlokban az oszlop már jelen van
- A `_delta_log` metaData akciója frissíti a sémanaplót

> **Megjegyzés:** `schema_mode="merge"` a `engine="rust"` (delta-rs natív writer) kapcsolóval érhető el ebben a verzióban.

In [ ]:
# Uj adag: loyalty_tier oszloppal bovitett sema
TIERS = ["Bronze", "Silver", "Gold", "Platinum"]
base3 = datetime(2024, 6, 1)
rows3 = []
for i in range(2501, 3001):
    rows3.append({
        "order_id":     i,
        "customer_id":  random.randint(1, 500),
        "product":      random.choice(CATEGORIES),
        "order_date":   (base3 + timedelta(days=random.randint(0, 180))).strftime("%Y-%m-%d"),
        "quantity":     random.randint(1, 10),
        "unit_price":   round(random.uniform(10, 500), 2),
        "order_status": random.choice(STATUSES),
        "loyalty_tier": random.choice(TIERS),
    })

df_v2 = pd.DataFrame(rows3)
df_v2["revenue"] = (df_v2["quantity"] * df_v2["unit_price"]).round(2)

# schema_mode="merge" csak engine="rust"-tal működik ebben a verzióban
write_deltalake(
    DELTA_PATH,
    pa.Table.from_pandas(df_v2, preserve_index=False),
    mode="append",
    schema_mode="merge",
    engine="rust",
)

print("Version 2 (schema evolution) irva.")
print()

# Aktualis sema
dt = DeltaTable(DELTA_PATH)
print("=== Aktualis Delta tabla sema ===")
for field in dt.schema().fields:
    print(f"  {field.name:<15} {str(field.type)}")

In [ ]:
# Ellenorzes: regi soroknal NULL, uj soroknal kitoltott loyalty_tier
dt = DeltaTable(DELTA_PATH)
df_all = dt.to_pandas()

null_tiers   = df_all["loyalty_tier"].isna().sum()
filled_tiers = df_all["loyalty_tier"].notna().sum()

print(f"Osszes sor:              {len(df_all):,}")
print(f"loyalty_tier = NULL:     {null_tiers:,}  (regi verziokbol)")
print(f"loyalty_tier kitoltott:  {filled_tiers:,}  (v2-bol)")
print()
print("loyalty_tier megoszlas (v2 soroknal):")
print(df_all["loyalty_tier"].value_counts().to_string())

## 5. rész: MERGE / Upsert

### SQL MERGE szemantika Python API-val

A MERGE az egyik legfontosabb Delta Lake funkció, mert lehetővé teszi:
- **WHEN MATCHED UPDATE**: létező sorok frissítése (pl. `order_status` változott)
- **WHEN NOT MATCHED INSERT**: új sorok hozzáadása
- Mindkét eset **egyetlen atomi tranzakcióban** hajtódik végre

**Adatmérnöki felhasználás:** CDC (Change Data Capture) alkalmazása –
a forrásrendszer változásait inkrementálisan alkalmazzuk a Delta táblán MERGE-zel,
nem teljes overwrite-tal.

In [ ]:
# Forrásadat a MERGE-hez:
# - 200 letező rendelés: order_status frissítve 'delivered'-re
# - 100 teljesen uj rendelés (order_id 3001-3100)

# 200 frissítes (mar letező order_id-k)
existing_ids = df_all["order_id"].sample(200, random_state=42).tolist()
updates = []
for oid in existing_ids:
    row = df_all[df_all["order_id"] == oid].iloc[0]
    updates.append({
        "order_id":     int(oid),
        "customer_id":  int(row["customer_id"]),
        "product":      row["product"],
        "order_date":   str(row["order_date"])[:10],
        "quantity":     int(row["quantity"]),
        "unit_price":   float(row["unit_price"]),
        "order_status": "delivered",
        "revenue":      float(row["revenue"]),
        "loyalty_tier": None,
    })

# 100 uj sor
base4 = datetime(2024, 12, 1)
for i in range(3001, 3101):
    updates.append({
        "order_id":     i,
        "customer_id":  random.randint(1, 500),
        "product":      random.choice(CATEGORIES),
        "order_date":   (base4 + timedelta(days=random.randint(0, 30))).strftime("%Y-%m-%d"),
        "quantity":     random.randint(1, 10),
        "unit_price":   round(random.uniform(10, 500), 2),
        "order_status": "pending",
        "revenue":      0.0,
        "loyalty_tier": random.choice(TIERS),
    })

source_df = pd.DataFrame(updates)
source_df["revenue"] = (source_df["quantity"] * source_df["unit_price"]).round(2)

print(f"MERGE forras: {len(updates):,} sor ({len(existing_ids)} update + 100 insert)")

# MERGE végrehajtása
(
    DeltaTable(DELTA_PATH)
    .merge(
        source=source_df,
        predicate="t.order_id = s.order_id",
        source_alias="s",
        target_alias="t",
    )
    .when_matched_update({
        "order_status": "s.order_status",
    })
    .when_not_matched_insert_all()
    .execute()
)

print("MERGE vegrehajtva.")

In [ ]:
# MERGE ellenorzese
dt = DeltaTable(DELTA_PATH)
df_merged = dt.to_pandas()

delivered_count = (df_merged["order_status"] == "delivered").sum()
new_rows = df_merged[df_merged["order_id"] >= 3001]

print("=== MERGE eredmeny ===")
print(f"Osszes sor:           {len(df_merged):,}")
print(f"'delivered' statuszu: {delivered_count:,}  "
      f"(volt: {(df_all['order_status'] == 'delivered').sum()})")
print(f"Uj sorok (id>=3001):  {len(new_rows):,}")
print()
print("order_status megoszlas:")
print(df_merged["order_status"].value_counts().to_string())
print()
print("=== Teljes history ===")
for h in dt.history():
    print(f"  v{h.get('version')}  {h.get('operation'):<15}  {str(h.get('timestamp'))[:19]}")

## 6. rész: DuckDB + Delta Lake

### Natív Delta olvasás SQL-lel

A DuckDB `delta` extension lehetővé teszi, hogy **SQL-lel közvetlenül olvassuk
a Delta táblákat**, anélkül hogy pandas DataFrame-be kellene tölteni:

```sql
INSTALL delta;
LOAD delta;
SELECT * FROM delta_scan('/path/to/delta_table');
```

**Előnyök:**
- Predicate pushdown: a WHERE feltétel a Parquet fájloknál szűr
- Columnar projection: csak a szükséges oszlopok olvasódnak be

**Time travel + DuckDB kombinálva:**
A DuckDB `delta_scan` egyelőre nem támogatja a `version` paramétert közvetlenül.
A megoldás: `DeltaTable(path, version=N).to_pandas()` → `con.register()` →
SQL lekérdezés a regisztrált nézetből. Ez kombinálja a delta-rs time travel
képességét a DuckDB analitikai erejével.

In [ ]:
# DuckDB delta extension telepítése és betöltése
con = duckdb.connect()
con.execute("INSTALL delta")
con.execute("LOAD delta")
print("DuckDB delta extension betoltve.")

# Analitikai lekérdezés közvetlenül a Delta tablából
result = con.execute(
    "SELECT product, order_status, "
    "COUNT(*) AS num_orders, "
    "ROUND(SUM(revenue), 0) AS total_revenue, "
    "ROUND(AVG(unit_price), 2) AS avg_price "
    "FROM delta_scan('" + DELTA_PATH + "') "
    "GROUP BY product, order_status "
    "ORDER BY product, total_revenue DESC"
).df()

print(f"\nBevétel termek x status szerint ({len(result)} sor):")
print(result.to_string(index=False))

In [ ]:
# Time travel DuckDB-bol: a Python API-val olvassuk a regi verziót,
# majd regisztraljuk DuckDB-be SQL lekérdezéshez
df_v0_again = DeltaTable(DELTA_PATH, version=0).to_pandas()
df_current  = DeltaTable(DELTA_PATH).to_pandas()

# DuckDB-be regisztrálva – SQL-lel lekérdezhető
con.register("orders_v0",      df_v0_again)
con.register("orders_current", df_current)

r0   = con.execute("SELECT COUNT(*) AS cnt, MIN(order_date) AS min_d, MAX(order_date) AS max_d FROM orders_v0").fetchone()
rcur = con.execute("SELECT COUNT(*) AS cnt, MIN(order_date) AS min_d, MAX(order_date) AS max_d FROM orders_current").fetchone()

print("=== DuckDB time travel osszehasonlitas ===")
print(f"  Version 0:  {r0[0]:>6,} sor  {r0[1]} -> {r0[2]}")
print(f"  Aktualis:   {rcur[0]:>6,} sor  {rcur[1]} -> {rcur[2]}")
print()

# Pelda: csak a v0 adatokon futtatott aggregacio (2022-2023)
result = con.execute("""
    SELECT product,
           COUNT(*)                   AS num_orders,
           ROUND(SUM(revenue), 0)     AS total_revenue
    FROM orders_v0
    GROUP BY product
    ORDER BY total_revenue DESC
""").df()
print("Bevétel termékenként (Version 0 – 2022–2023 adatok):")
print(result.to_string(index=False))

## Összefoglalás

| Képesség | Plain Parquet | Delta Lake |
|----------|:---:|:---:|
| ACID írás | ✗ | ✓ |
| Time travel | ✗ | ✓ |
| Schema evolution | kézi | `schema_mode="merge"` |
| MERGE / Upsert | ✗ | ✓ |
| Transaction log | ✗ | `_delta_log/` JSON |
| DuckDB olvasás | ✓ (`read_parquet`) | ✓ (`delta_scan`) |
| Spark függőség | ✗ | ✗ (delta-rs / deltalake csomag) |

**Mikor válasszuk a Delta Lake-et?**
- Ha több pipeline-ból is írnak ugyanabba a táblába (konkurens írók)
- Ha CDC alapú inkrementális betöltés kell
- Ha visszaállíthatóság és audit trail fontos
- Ha a séma idővel változhat

**Alternatív Open Table Formats:** Apache Iceberg (Netflix), Apache Hudi (Uber)
– hasonló képességek, eltérő architektúra.